In [3]:
# Setup
import torch
import copy
import json

from utils.data_reader import load_and_prepare_time_series_data
from utils.evaluator import Evaluator
from models.baseline_models import LSTMModel, BiLSTMModel, GRUModel
from models.custom_models import MGSSMModel, MGSSMsModel, ExtendedMGSSMsModel

In [2]:

selected_country_codes = ['US', 'IN', 'BR', 'FR', 'DE',
                         'GB', 'RU', 'IT', 'TR', 'ES',
                         'VN', 'AR', 'AU', 'AT', 'BD',
                         'BE', 'BG', 'CA', 'CL', 'CN',
                         'CU', 'DK', 'FI', 'GE', 'GR',
                         'ID', 'JP', 'JO', 'KE', 'KR',
                         'LR', 'MY','ML', 'MX', 'NL',
                         'NO', 'PH','SE', 'CH', 'TH']

# Fetch and prepare the data for each country code
with open("/home/theppawan/nn-models/data/COVID19_url_data.json", "r") as f:
    dataset = json.load(f)

train_loader_dist = {}
val_loader_dist = {}
test_loader_dist = {}
scaler_dist = {}
for key in selected_country_codes:
    train_loader, val_loader, test_loader, scaler = load_and_prepare_time_series_data(
        filepath_or_url = dataset['region'].format(region=key),
        target_column=['cumulative_confirmed'],
        date_column="date",
        seq_length=14,
        batch_size=64,
        train_split=0.8,
        fill_missing=True)
    train_loader_dist[key] = train_loader
    val_loader_dist[key] = val_loader
    test_loader_dist[key] = test_loader
    scaler_dist[key] = scaler

# Setup model configurations
model_config_dict = {
    "Baseline LSTM": LSTMModel(input_size=1, hidden_size=256, num_layers=1, output_size=1),
    "Baseline BiLSTM": BiLSTMModel(input_size=1, hidden_size=256, num_layers=1, output_size=1),
    "Baseline GRU": GRUModel(input_size=1, hidden_size=128, num_layers=1, output_size=1),
    "MGSSM": MGSSMModel(input_size=1, hidden_size=64, num_layers=1, output_size=1, gate_size=32),
    "MGSSMs": MGSSMsModel(input_size=1, hidden_size=64, num_layers=1, output_size=1, gate_size=32),
    "ExtendedMGSSMs": ExtendedMGSSMsModel(input_size=1, hidden_size=64, num_layers=1, output_size=1, gate_size=32, p=2)
}

Fetching data from: https://storage.googleapis.com/covid19-open-data/v3/location/US.csv
Sorting data chronologicall by column: date
Succesfully loaded 991 sequential data points.
Prepared Training batches: 13 | Validation batches: 3
Fetching data from: https://storage.googleapis.com/covid19-open-data/v3/location/IN.csv
Sorting data chronologicall by column: date
Succesfully loaded 991 sequential data points.
Prepared Training batches: 13 | Validation batches: 3
Fetching data from: https://storage.googleapis.com/covid19-open-data/v3/location/BR.csv
Sorting data chronologicall by column: date
Succesfully loaded 991 sequential data points.
Prepared Training batches: 13 | Validation batches: 3
Fetching data from: https://storage.googleapis.com/covid19-open-data/v3/location/FR.csv
Sorting data chronologicall by column: date
Succesfully loaded 991 sequential data points.
Prepared Training batches: 13 | Validation batches: 3
Fetching data from: https://storage.googleapis.com/covid19-open-data

In [6]:
trained_model_dict = {key: {} for key in selected_country_codes}
# loading the checkpointed models
for model_name, model_config in model_config_dict.items():
    # analyze the model from the checkpoint
    for key in selected_country_codes:
        country_specific_model = copy.deepcopy(model_config)
        country_specific_model.load_state_dict(torch.load(f"checkpoints/best_{model_config.__class__.__name__}_{key}.pth"))
        trained_model_dict[key][model_name] = country_specific_model

evaluator = Evaluator()
for key in selected_country_codes:
    val_loader = val_loader_dist[key]
    scaler = scaler_dist[key]
    evaluator.compare_models(trained_model_dict[key], val_loader, scaler)

Evaluating model: Baseline LSTM...
Evaluating model: Baseline BiLSTM...
Evaluating model: Baseline GRU...
Evaluating model: MGSSM...
Evaluating model: MGSSMs...
Evaluating model: ExtendedMGSSMs...
Evaluating model: Baseline LSTM...
Evaluating model: Baseline BiLSTM...
Evaluating model: Baseline GRU...
Evaluating model: MGSSM...
Evaluating model: MGSSMs...
Evaluating model: ExtendedMGSSMs...
Evaluating model: Baseline LSTM...
Evaluating model: Baseline BiLSTM...
Evaluating model: Baseline GRU...
Evaluating model: MGSSM...
Evaluating model: MGSSMs...
Evaluating model: ExtendedMGSSMs...
Evaluating model: Baseline LSTM...
Evaluating model: Baseline BiLSTM...
Evaluating model: Baseline GRU...
Evaluating model: MGSSM...
Evaluating model: MGSSMs...
Evaluating model: ExtendedMGSSMs...
Evaluating model: Baseline LSTM...
Evaluating model: Baseline BiLSTM...
Evaluating model: Baseline GRU...
Evaluating model: MGSSM...
Evaluating model: MGSSMs...
Evaluating model: ExtendedMGSSMs...
Evaluating mode

In [9]:
performance_data = evaluator.build_performance_dataframe(nested_models_dict=trained_model_dict, data_loaders_dict=val_loader_dist, scalers_dict=scaler_dist, target_metric="R2")
performance_data

Evaluating models for dataset: US...
Evaluating models for dataset: IN...
Evaluating models for dataset: BR...
Evaluating models for dataset: FR...
Evaluating models for dataset: DE...
Evaluating models for dataset: GB...
Evaluating models for dataset: RU...
Evaluating models for dataset: IT...
Evaluating models for dataset: TR...
Evaluating models for dataset: ES...
Evaluating models for dataset: VN...
Evaluating models for dataset: AR...
Evaluating models for dataset: AU...
Evaluating models for dataset: AT...
Evaluating models for dataset: BD...
Evaluating models for dataset: BE...
Evaluating models for dataset: BG...
Evaluating models for dataset: CA...
Evaluating models for dataset: CL...
Evaluating models for dataset: CN...
Evaluating models for dataset: CU...
Evaluating models for dataset: DK...
Evaluating models for dataset: FI...
Evaluating models for dataset: GE...
Evaluating models for dataset: GR...
Evaluating models for dataset: ID...
Evaluating models for dataset: JP...
E

,Baseline LSTM,Baseline BiLSTM,Baseline GRU,MGSSM,MGSSMs,ExtendedMGSSMs
US,0.998987,0.999095,0.999653,0.999078,0.999079,0.999768
IN,0.999956,0.999901,0.999873,0.999625,0.999382,0.999017
BR,0.999805,0.999892,0.999766,0.997106,0.999152,0.999774
FR,0.946196,0.972084,0.954177,0.980007,0.975625,0.977757
DE,0.999351,0.999784,0.998977,0.996987,0.998107,0.999166
GB,0.999660,0.999464,0.998299,0.998840,0.999098,0.999757
RU,0.999553,0.999692,0.998365,0.999293,0.999230,0.999655
IT,0.999858,0.999890,0.998575,0.999887,0.999742,0.999899
TR,0.989962,0.995821,0.988271,0.995270,0.994791,0.994919
ES,0.999318,0.999571,0.997026,0.996911,0.999102,0.999334


In [10]:
friedman_statistic, p_value = evaluator.friedman_test(performance_data)
p_value

np.float64(0.0005822785735293359)